In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [24]:
# benchmark dataset
# 目标: 12个tag进行rag，3秒内
import random

from entropy.domain.services.tag_checker import TagChecker


input_data = """
shimmering, pleading, soaked, slightly open mouth, thin straps, sheer dress, blushing, shoulderless dress,
seifuku, japanese school uniform, bubbles, knitwear,
low angle, daydreaming, wild life, surrounded by bubbles, sunbeams, cross-legged, flowing dress,
adjusting glove, rim light, tyndall effect, floating flower petals, misty,
magical atmosphere, motes, thoughtful, dangling legs, cinematic lighting, purple and gold theme,
reading book, white clouds, large bubbles, scenic
"""

input_data = TagChecker.extract_all_tags(input_data)
input_data = list(set(input_data))
input_data = sorted(input_data)

input_data = random.Random(3).sample(input_data, k=12)

assert len(input_data) == 12

print(",".join(input_data))

motes,flowing dress,shimmering,thin straps,rim light,wild life,scenic,reading book,bubbles,thoughtful,adjusting glove,surrounded by bubbles


In [25]:
from entropy.domain.services.rag_service import RagService

In [35]:
# 非batch

result1 = []

for invalid_tag in input_data:
    tags, scores = RagService.do_rag(query_text=invalid_tag, recall_count=20, rerank_output=10)
    result1.append(tags)

In [36]:
# batch
batch_rag_output = RagService.batch_rag(query_text_list=input_data,recall_count=10, rerank_output=0)

result2 = [tags for tags, scores in batch_rag_output]

In [37]:
import json



for (invalid_tag, r1, r2) in zip(input_data, result1, result2):
    if not json.dumps(r1) == json.dumps(r2):
        print(f"invalid danbooru tag: {invalid_tag}")
        print("rag result 1:", r1)
        print("rag result 2:", r2)

invalid danbooru tag: motes
rag result 1: ['moth', 'mot', 'mota', 'mots', 'motida', 'mentos', 'mootor', 'moltres', 'mintes', 'moroes']
rag result 2: ['moltres', 'mots', 'mot', 'moss', 'moroes', 'mos', 'mota', 'moth', 'motatei', 'mintes']
invalid danbooru tag: flowing dress
rag result 1: ['floral dress', 'water dress', 'wet dress', 'dress flower', 'floating clothes', 'frilled dress', 'holding dress', 'dress flip', 'multicolored dress', 'dress']
rag result 2: ['water dress', 'wet dress', 'dress flower', 'floral dress', 'dress', 'layered dress', 'feather dress', 'multicolored dress', 'frilled dress', 'floating clothes']
invalid danbooru tag: shimmering
rag result 1: ['shimmer', 'shining', 'glowing skin', 'glowing', 'the shining', 'shiny skin', 'sunset shimmer', 'glowing jewelry', 'glowing eyes', 'swirling']
rag result 2: ['shimmer', 'sunset shimmer', 'shining', 'glowing', 'the shining', 'shining star', 'shining wind', 'glowing skin', 'sparkling eyes', 'glowing hot']
invalid danbooru tag: 

In [ ]:
"""
我试图从方法1改为方法2，请帮我看看效果是否有明显下降。从2个角度：

1. 这个是检测到无效tag，对用户输出guess you like tags，帮助用户找到原本想要找的哪个

2. 除了将无效变为有效之外，tag也能提供灵感

"""
print()

## 实验结论

### rerank 一定好吗
recall -> 500 -> rerank -> 10

recall -> 10

前者效果差，后者好（经过gemini和豆包反复看case）

也可能是因为recall没有卡阈值造成的，但我懒得卡，不想尝试了

### 尝试优化方法1

recall -> 20 -> rerank -> 10 // 500改成20

recall -> 10

实验结果：还是方法2更好。

### 最终结论
所以最终结论是：这个耗时的reranker费力不讨好。

既然这样，bm25也没必要支持了，因为bm25肯定比embedding差；bm25+reranker也一定比embedding差。用户花30分钟embedding建库没啥大问题。

In [ ]:
#